# Session 09 — Object Detection

**CVI4IC · Summer Semester 2026 · FH Upper Austria**

> Today we move from *“what is in this image?”* to *“what is in this image, and where?”*
> We first **build a tiny anchor-box detector from scratch** on a synthetic fruit dataset — to feel how anchors, objectness and box regression work — then meet three production detectors:
>
> - **Part A** — a from-scratch anchor-box network (illustrative).
> - **Part B — YOLO11** — Ultralytics, the anchor-free one-stage workhorse.
> - **Part C — RT-DETR** — Baidu's real-time DETR; no NMS, transformer queries.
> - **Part D — VLM** — Gemma 4: the LM writes the boxes out as **JSON**.
>
> Everything is exercised on the same kind of images so you can compare.

> **Colab tip:** Runtime → Change runtime type → **GPU** before running. We use a small T4 happily.


## 0. Setup

The first cell installs the three libraries we need that aren't already in Colab:
* `ultralytics` — YOLO11 + auxiliary trainers.
* `transformers` (upgraded) — HuggingFace, for RT-DETR and Gemma 4.
* `supervision` — clean drawing utilities (boxes, labels, NMS).

> We only force-upgrade `transformers` (Gemma 4 needs it); everything else installs without `-U` so Colab's preinstalled packages (e.g. pillow) stay as-is.
>
> ⚠️ **You must restart the runtime after the install cell** — see the note right below it.


In [ ]:
# Gemma 4 needs a recent transformers; scope -U to it so Colab's pillow/gradio stay intact.
!pip install -q -U transformers
!pip install -q ultralytics supervision accelerate

> 🔁 **Restart the runtime now.** Upgrading `transformers` in a running kernel only takes effect after a restart:
> **Runtime → Restart session** (Colab may also pop up a "RESTART RUNTIME" button — click it).
>
> After restarting, **skip this install cell** and run everything from the next cell onward. If you hit an import error or *"gemma4 architecture not recognized"*, a stale `transformers` is almost always the cause.


In [ ]:
import os, io, time, math, urllib.request, zipfile, random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "·  torch", torch.__version__)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 110

## 1. Sample images

We pull two multi-object scenes from our course data repo **[Digital-Media/cv_data](https://github.com/Digital-Media/cv_data)**:
* `street.jpg` — people + vehicles (COCO classes) for **YOLO / RT-DETR**.
* `fruits.jpg` — a bowl/pile of fruit for the **Gemma** demo (and YOLO too: *apple/banana/orange* are COCO classes).


In [ ]:
RAW = "https://raw.githubusercontent.com/Digital-Media/cv_data/main/object_detection"
SAMPLE_URLS = {
    "street.jpg": f"{RAW}/street.jpg",   # people + vehicles (COCO) — YOLO / RT-DETR
    "fruits.jpg": f"{RAW}/fruits.jpg",   # bowl/pile of fruit — Gemma (and YOLO: apple/banana/orange are COCO)
}

DATA = Path("data09"); DATA.mkdir(exist_ok=True)

for fname, url in SAMPLE_URLS.items():
    p = DATA / fname
    if not p.exists():
        try:
            urllib.request.urlretrieve(url, p)
            print(f"Saved {fname}")
        except Exception as e:
            print(f"Couldn't download {fname}: {e}")

# Show what we have
imgs = [Image.open(p).convert("RGB") for p in DATA.glob("*.jpg") if p.is_file()]
fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
if len(imgs) == 1: axes = [axes]
for ax, img, p in zip(axes, imgs, sorted(DATA.glob("*.jpg"))):
    ax.imshow(img); ax.set_title(p.name); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Part A — Anchor boxes from scratch (illustrative)

Before we call pretrained models, let's *build the core idea ourselves* on a synthetic dataset, so anchors stop being magic. We paste fruit crops at random positions and scales, record their bounding boxes, then train a tiny CNN that, for every cell of a coarse grid, predicts:

* **objectness** — is the centre of a fruit in this cell?
* **box regression** — four offsets `(tx, ty, tw, th)` that warp the cell's **anchor** onto the true box.

This is exactly the recipe from the lecture (anchor grid → assignment → offset regression → NMS), just shrunk to one class, one anchor per cell, and one scale.

> 12 epochs: a few seconds on a GPU, ~4–6 min on CPU. It won't win COCO — it's here to *teach*.


In [ ]:
import math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import nms
from PIL import Image
from pathlib import Path

# Grab a handful of varied fruit crops from Fruits-360 (white background → we mask it out).
import subprocess
subprocess.run("git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git",
               shell=True, capture_output=True)
BASE = Path("fruits-360-100x100/Training")
WANT = ["Apple Braeburn 1", "Banana 1", "Cucumber 1", "Plum 1", "Cherry 1"]
fruit_crops = []
for cls in WANT:
    d = BASE / cls
    if d.exists():
        fruit_crops += [Image.open(p).convert("RGB") for p in sorted(d.glob("*.jpg"))[:15]]
if not fruit_crops:                                   # fallback: first few class folders
    for d in sorted(BASE.glob("*"))[:5]:
        fruit_crops += [Image.open(p).convert("RGB") for p in sorted(d.glob("*.jpg"))[:15]]
print("Loaded", len(fruit_crops), "fruit crops")

def paste_fruit(canvas, crop, x, y, size):
    crop = crop.resize((size, size))
    arr = np.array(crop)
    mask = Image.fromarray(((arr.sum(2) < 740) * 255).astype("uint8"))  # drop near-white bg
    canvas.paste(crop, (x, y), mask)

In [ ]:
IMG = 128                       # canvas size

class SynthFruits(Dataset):
    """On-the-fly images with 1-3 fruits at random positions / scales + their boxes."""
    def __init__(self, n=320, max_objs=3):
        self.n, self.max_objs = n, max_objs
    def __len__(self): return self.n
    def __getitem__(self, idx):
        canvas = Image.new("RGB", (IMG, IMG), tuple(np.random.randint(40, 210, 3).tolist()))
        boxes = []
        for _ in range(random.randint(1, self.max_objs)):
            crop = random.choice(fruit_crops)
            size = random.randint(28, 56)
            x, y = random.randint(0, IMG - size), random.randint(0, IMG - size)
            paste_fruit(canvas, crop, x, y, size)
            boxes.append([x, y, x + size, y + size])
        img = torch.from_numpy(np.array(canvas)).permute(2, 0, 1).float() / 255.
        return img, torch.tensor(boxes, dtype=torch.float32)

def collate(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

# Peek at a few synthetic samples
ds = SynthFruits()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax in axes:
    img, bx = ds[random.randint(0, len(ds)-1)]
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
    for x1, y1, x2, y2 in bx.tolist():
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", lw=2))
plt.suptitle("Synthetic dataset — fruits pasted at random positions, red = ground-truth box")
plt.tight_layout(); plt.show()

### Encoding boxes onto the anchor grid

We tile the 128×128 image with a coarse **8×8 grid** (stride 16). Each cell owns one square **anchor** (44 px). A cell is **positive** if a ground-truth box centre falls inside it; its regression target is the offset from the anchor to that box — `tx, ty` inside the cell, `tw, th` in log-scale (exactly the scale-invariant target from the slides).


In [ ]:
STRIDE = 16
G = IMG // STRIDE          # 8x8 grid
ANCHOR = 44.0              # one square anchor per cell (px)

def encode(boxes):
    """Boxes (N,4 xyxy) -> obj (G,G) and reg targets (G,G,4)."""
    obj = torch.zeros(G, G)
    reg = torch.zeros(G, G, 4)
    for x1, y1, x2, y2 in boxes.tolist():
        cx, cy, w, h = (x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1
        gx, gy = min(int(cx // STRIDE), G-1), min(int(cy // STRIDE), G-1)
        obj[gy, gx] = 1.0
        reg[gy, gx] = torch.tensor([cx/STRIDE - gx, cy/STRIDE - gy,
                                     math.log(w/ANCHOR + 1e-6), math.log(h/ANCHOR + 1e-6)])
    return obj, reg

# Visualise the assignment on one sample
img, bx = ds[0]
obj_t, _ = encode(bx)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img.permute(1, 2, 0).numpy())
for i in range(G + 1):
    ax.axhline(i*STRIDE, color="w", lw=.5, alpha=.5); ax.axvline(i*STRIDE, color="w", lw=.5, alpha=.5)
ys, xs = torch.where(obj_t > 0.5)
for gy, gx in zip(ys.tolist(), xs.tolist()):
    ax.add_patch(patches.Rectangle((gx*STRIDE, gy*STRIDE), STRIDE, STRIDE, facecolor="lime", alpha=.35))
for x1, y1, x2, y2 in bx.tolist():
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", lw=2))
ax.set_title("Anchor assignment — green = positive cell (holds a GT centre)"); ax.axis("off")
plt.show()

### A tiny detector network

Four conv-blocks take the image down to the 8×8 grid; a 1×1 head outputs **5 numbers per cell**: 1 objectness logit + 4 box offsets.


In [ ]:
class TinyDetector(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.backbone = nn.Sequential(block(3, 16), block(16, 32), block(32, 64), block(64, 128))  # /16
        self.head = nn.Conv2d(128, 5, 1)              # [obj, tx, ty, tw, th] per cell
    def forward(self, x):
        return self.head(self.backbone(x))            # (B, 5, G, G)

model_scratch = TinyDetector().to(device)
print(sum(p.numel() for p in model_scratch.parameters())/1e3, "k parameters")

### Train

Objectness uses binary cross-entropy over **all** cells; the box loss is a smooth-L1 on `(tx, ty, tw, th)` for **positive** cells only.


In [ ]:
opt = torch.optim.Adam(model_scratch.parameters(), lr=2e-3)
loader = DataLoader(SynthFruits(320), batch_size=16, collate_fn=collate, shuffle=True)

for epoch in range(12):
    model_scratch.train(); running = 0.0
    for imgs, tgts in loader:
        imgs = imgs.to(device); B = imgs.size(0)
        obj_t = torch.zeros(B, G, G); reg_t = torch.zeros(B, G, G, 4)
        for i, bx in enumerate(tgts):
            obj_t[i], reg_t[i] = encode(bx)
        obj_t, reg_t = obj_t.to(device), reg_t.to(device)

        out = model_scratch(imgs)                     # B,5,G,G
        obj_p = out[:, 0]
        reg_p = out[:, 1:].permute(0, 2, 3, 1)        # B,G,G,4
        loss_obj = F.binary_cross_entropy_with_logits(obj_p, obj_t)
        pos = obj_t > 0.5
        if pos.any():
            p, t = reg_p[pos], reg_t[pos]
            loss_box = F.smooth_l1_loss(torch.sigmoid(p[:, :2]), t[:, :2]) + \
                       F.smooth_l1_loss(p[:, 2:], t[:, 2:])
        else:
            loss_box = torch.zeros((), device=device)
        loss = loss_obj + 2.0 * loss_box
        opt.zero_grad(); loss.backward(); opt.step(); running += loss.item()
    print(f"epoch {epoch+1}: loss {running/len(loader):.3f}")

### Decode → NMS → look

We threshold objectness, invert the regression to pixel boxes, and run plain **NMS** (the exact post-process the slides described). Dashed yellow = ground truth, solid green = prediction.


In [ ]:
@torch.no_grad()
def detect_scratch(img_t, thresh=0.5, iou=0.3):
    model_scratch.eval()
    out = model_scratch(img_t[None].to(device))[0]    # 5,G,G
    obj = torch.sigmoid(out[0]); reg = out[1:].permute(1, 2, 0)
    boxes, scores = [], []
    ys, xs = torch.where(obj > thresh)
    for gy, gx in zip(ys.tolist(), xs.tolist()):
        tx, ty, tw, th = reg[gy, gx]
        cx = (gx + torch.sigmoid(tx)) * STRIDE; cy = (gy + torch.sigmoid(ty)) * STRIDE
        w = ANCHOR * torch.exp(tw); h = ANCHOR * torch.exp(th)
        boxes.append([float(cx-w/2), float(cy-h/2), float(cx+w/2), float(cy+h/2)])
        scores.append(float(obj[gy, gx]))
    if not boxes:
        return [], []
    b, s = torch.tensor(boxes), torch.tensor(scores)
    keep = nms(b, s, iou)
    return b[keep].tolist(), s[keep].tolist()

test = SynthFruits(8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), range(8)):
    img, gt = test[idx]
    preds, scs = detect_scratch(img, 0.5)
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
    for x1, y1, x2, y2 in gt.tolist():
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="yellow", lw=1.2, ls="--"))
    for (x1, y1, x2, y2), s in zip(preds, scs):
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", lw=2))
        ax.text(x1, max(0, y1-3), f"{s:.2f}", color="black", fontsize=8,
                bbox=dict(facecolor="lime", alpha=.7, edgecolor="none", pad=1))
    ax.set_title(f"{len(preds)} detection(s)", fontsize=9)
plt.suptitle("Tiny anchor-box detector — dashed yellow = GT, solid green = prediction")
plt.tight_layout(); plt.show()

**Takeaway.** That's the whole anchor-box machinery in ~60 lines: a grid of priors, centre-based assignment, log-space size regression, and NMS to clean up. **YOLO** (next) generalises this with a real backbone, multiple scales (FPN), an anchor-free head, and far more training — but the skeleton is what you just built.


## 3. Part B — YOLO11 with Ultralytics

The Ultralytics package wraps detector training, inference, and visualization. A pretrained `yolo11n.pt` model is ~6 MB and ships with COCO weights (80 classes).


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")           # nano variant — small + fast
print(model.info(verbose=False))
print("Predicting on", DATA / "street.jpg")
res = model.predict(source=str(DATA / "street.jpg"), conf=0.25, verbose=False)[0]
print("Detections:", len(res.boxes))

### Look at the predictions

In [ ]:
def show_boxes_pil(image_path, boxes, labels=None, scores=None, color="lime", ax=None):
    """Draw [xyxy] boxes on an image with optional labels."""
    img = Image.open(image_path).convert("RGB")
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img); ax.axis("off")
    W, H = img.size
    for i, b in enumerate(boxes):
        x1, y1, x2, y2 = b
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels is not None:
            txt = labels[i]
            if scores is not None: txt += f" {scores[i]:.2f}"
            ax.text(x1, max(0, y1-4), txt, color="black",
                    fontsize=9, bbox=dict(facecolor=color, alpha=0.7, edgecolor="none", pad=1))
    return ax

# Extract YOLO predictions
boxes  = res.boxes.xyxy.cpu().numpy()
clsids = res.boxes.cls.int().cpu().tolist()
confs  = res.boxes.conf.cpu().tolist()
names  = [res.names[i] for i in clsids]

ax = show_boxes_pil(DATA / "street.jpg", boxes, names, confs, color="lime")
ax.set_title("YOLO11n detections (conf ≥ 0.25)")
plt.show()

### NMS in action — before vs after

YOLO does NMS internally. To see *why* it matters, we ask the model to emit all per-anchor predictions before NMS (`agnostic_nms=False`, high IoU threshold) and compare to the cleaned output.


In [ ]:
# Get RAW predictions with a permissive NMS (lots of overlapping survivors)
res_raw = model.predict(source=str(DATA / "street.jpg"), conf=0.05, iou=0.95, verbose=False)[0]

# Standard (default) predictions — NMS already applied
res_nms = model.predict(source=str(DATA / "street.jpg"), conf=0.25, iou=0.5, verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
show_boxes_pil(DATA / "street.jpg",
               res_raw.boxes.xyxy.cpu().numpy(),
               labels=[res_raw.names[int(c)] for c in res_raw.boxes.cls],
               scores=res_raw.boxes.conf.cpu().tolist(),
               color="red", ax=axes[0])
axes[0].set_title(f"Before NMS — {len(res_raw.boxes)} boxes")

show_boxes_pil(DATA / "street.jpg",
               res_nms.boxes.xyxy.cpu().numpy(),
               labels=[res_nms.names[int(c)] for c in res_nms.boxes.cls],
               scores=res_nms.boxes.conf.cpu().tolist(),
               color="lime", ax=axes[1])
axes[1].set_title(f"After NMS — {len(res_nms.boxes)} boxes")
plt.tight_layout(); plt.show()

### How fast is it?

YOLO11n on a T4 should comfortably hit > 100 fps. Let's measure.


In [ ]:
# warm-up + timing
for _ in range(3): model.predict(source=str(DATA / "street.jpg"), verbose=False)

t0 = time.time(); N = 30
for _ in range(N):
    _ = model.predict(source=str(DATA / "street.jpg"), verbose=False)
dt = (time.time() - t0) / N
print(f"YOLO11n: {1/dt:.1f} fps  ({dt*1000:.1f} ms / image)")

### Fine-tuning YOLO on a tiny custom dataset

YOLO11's COCO weights already know "apple", so fine-tuning to find apples wouldn't prove much. Instead we teach it a **brand-new class it has never seen: "pen"**. We *synthesise* the data — draw pens (rounded barrel + nib + cap) at random sizes, colours, positions and **rotations**, and record a tight box for each. A rotated thin object is also a far better localisation test than a square.

> Fully self-contained — no dataset download. ~150 train / 30 val images.


In [ ]:
from PIL import ImageDraw

PEN = Path("yolo_pens")
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    (PEN / sub).mkdir(parents=True, exist_ok=True)
CANVAS = 320

def rand_color():
    return tuple(random.randint(30, 230) for _ in range(3))

def draw_pen(length, thick):
    """Draw a horizontal pen on a transparent RGBA tile (barrel + dark nib + cap)."""
    pad = thick + 4
    im = Image.new("RGBA", (length + 2*pad, thick + 2*pad), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)
    color = rand_color()
    x0, y0, x1, y1 = pad, pad, pad + length, pad + thick
    d.rounded_rectangle([x0 + thick, y0, x1 - thick, y1], radius=thick // 2, fill=color + (255,))   # barrel
    d.polygon([(x1 - thick, y0), (x1 - thick, y1), (x1, (y0 + y1) // 2)], fill=(35, 35, 35, 255))    # nib
    cap = tuple(max(0, c - 70) for c in color)
    d.rounded_rectangle([x0, y0, x0 + thick + 2, y1], radius=thick // 2, fill=cap + (255,))          # cap
    return im

def make_sample(out_img, out_label, n_range=(1, 4)):
    canvas = Image.new("RGB", (CANVAS, CANVAS), rand_color())
    dd = ImageDraw.Draw(canvas)
    for _ in range(random.randint(5, 12)):                       # distractor squares
        x, y = random.randint(0, CANVAS-18), random.randint(0, CANVAS-18)
        dd.rectangle([x, y, x + random.randint(8, 18), y + random.randint(8, 18)], fill=rand_color())
    rows = []
    for _ in range(random.randint(*n_range)):
        pen = draw_pen(random.randint(70, 140), random.randint(12, 22))
        pen = pen.rotate(random.uniform(0, 360), expand=True, resample=Image.BICUBIC)
        pw, ph = pen.size
        if pw >= CANVAS or ph >= CANVAS:
            continue
        x, y = random.randint(0, CANVAS - pw), random.randint(0, CANVAS - ph)
        canvas.paste(pen, (x, y), pen)
        alpha = np.array(pen.split()[-1])                         # tight box from the rotated alpha
        ys, xs = np.where(alpha > 10)
        if len(xs) == 0:
            continue
        bx1, by1, bx2, by2 = xs.min()+x, ys.min()+y, xs.max()+x, ys.max()+y
        cx, cy = (bx1+bx2)/2/CANVAS, (by1+by2)/2/CANVAS
        rows.append(f"0 {cx:.4f} {cy:.4f} {(bx2-bx1)/CANVAS:.4f} {(by2-by1)/CANVAS:.4f}")
    canvas.save(out_img)
    out_label.write_text("\n".join(rows))

random.seed(0)
for i in range(150):
    make_sample(PEN / "images/train" / f"t{i:03d}.jpg", PEN / "labels/train" / f"t{i:03d}.txt")
for i in range(30):
    make_sample(PEN / "images/val"   / f"v{i:03d}.jpg", PEN / "labels/val"   / f"v{i:03d}.txt")

YAML = PEN / "dataset.yaml"
YAML.write_text(f"""path: {PEN.resolve()}
train: images/train
val:   images/val
names:
  0: pen
""")
print("Wrote", YAML, "·", len(list((PEN/'images/train').glob('*.jpg'))), "train /",
      len(list((PEN/'images/val').glob('*.jpg'))), "val images")

In [ ]:
# Sanity-check one synthetic sample with its labels
sample_img = sorted((PEN / "images/train").glob("*.jpg"))[0]
lbl = (PEN / "labels/train" / (sample_img.stem + ".txt")).read_text().strip().splitlines()
img = Image.open(sample_img); W, H = img.size
boxes = []
for line in lbl:
    _, cx, cy, w, h = map(float, line.split())
    boxes.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H])
ax = show_boxes_pil(sample_img, boxes, ["pen"]*len(boxes), color="orange")
ax.set_title(f"Synthetic training sample ({len(boxes)} pen(s))")
plt.show()

**Train.** A from-scratch class needs a fair few epochs before the model becomes *confident* (not just accurate). We give it **60 epochs** — ~3-4 min on a T4. Watch `mAP50` climb in the log.

> ⚠️ Quick fine-tunes on tiny synthetic data are often **under-confident**: the boxes are in the right place but scores stay low. That's why the prediction cell below **auto-lowers the confidence threshold** until detections appear, and prints which threshold it used. With more epochs/data the usable threshold rises toward the usual 0.25.


In [ ]:
# Fine-tune YOLO11n to detect 'pen'. 60 epochs on the synthetic set (~3-4 min on a T4).
model_ft = YOLO("yolo11n.pt")
model_ft.train(data=str(YAML), epochs=60, imgsz=CANVAS, batch=16,
               plots=False, verbose=False, exist_ok=True, seed=0)
metrics = model_ft.val(verbose=False)
print(f"\nPen-detector mAP@50   = {metrics.box.map50:.3f}")
print(f"Pen-detector mAP50-95 = {metrics.box.map:.3f}")

In [ ]:
# Predict on held-out val images. Auto-pick a confidence threshold that actually shows boxes
# (quick fine-tunes can be under-confident — the boxes are right, the scores are just low).
val_imgs = sorted((PEN / "images/val").glob("*.jpg"))[:4]

def total_at(conf):
    return sum(len(model_ft.predict(source=str(p), conf=conf, imgsz=CANVAS, verbose=False)[0].boxes)
               for p in val_imgs)

conf_show = next((c for c in [0.25, 0.15, 0.10, 0.05, 0.02, 0.01] if total_at(c) > 0), 0.01)
print(f"Showing detections at conf = {conf_show}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, val_imgs):
    r = model_ft.predict(source=str(p), conf=conf_show, imgsz=CANVAS, verbose=False)[0]
    show_boxes_pil(p, r.boxes.xyxy.cpu().numpy(),
                   ["pen"]*len(r.boxes), r.boxes.conf.cpu().tolist(),
                   color="lime", ax=ax)
    ax.set_title(f"{p.name} — {len(r.boxes)} det @ {conf_show}")
plt.tight_layout(); plt.show()

## 4. Part C — RT-DETR via HuggingFace

RT-DETR (Baidu, 2024) is a DETR variant designed for real-time use. It removes NMS, uses a hybrid encoder, and reaches YOLO-class speed.

We use the `PekingU/rtdetr_v2_r18vd` checkpoint — the small student model.


In [ ]:
from transformers import RTDetrImageProcessor, RTDetrForObjectDetection

rt_id = "PekingU/rtdetr_v2_r18vd"
rt_proc  = RTDetrImageProcessor.from_pretrained(rt_id)
rt_model = RTDetrForObjectDetection.from_pretrained(rt_id).to(device).eval()
print(f"RT-DETR loaded ({sum(p.numel() for p in rt_model.parameters())/1e6:.1f} M params)")

In [ ]:
def rtdetr_predict(image_path, threshold=0.3):
    img = Image.open(image_path).convert("RGB")
    inputs = rt_proc(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = rt_model(**inputs)
    target = torch.tensor([img.size[::-1]]).to(device)  # (H, W)
    res = rt_proc.post_process_object_detection(outputs, target_sizes=target, threshold=threshold)[0]
    boxes  = res["boxes"].cpu().numpy()
    scores = res["scores"].cpu().tolist()
    labels = [rt_model.config.id2label[i.item()] for i in res["labels"]]
    return boxes, labels, scores

boxes, names, confs = rtdetr_predict(DATA / "street.jpg", threshold=0.3)
print(f"RT-DETR found {len(boxes)} objects")
ax = show_boxes_pil(DATA / "street.jpg", boxes, names, confs, color="cyan")
ax.set_title("RT-DETR predictions"); plt.show()

### Side-by-side: YOLO11 vs RT-DETR

Same images, both detectors. Look for where they agree and where they disagree.


In [ ]:
def yolo_predict(image_path, threshold=0.25):
    res = model.predict(source=str(image_path), conf=threshold, verbose=False)[0]
    boxes  = res.boxes.xyxy.cpu().numpy()
    names  = [res.names[int(c)] for c in res.boxes.cls]
    confs  = res.boxes.conf.cpu().tolist()
    return boxes, names, confs

paths = [DATA / "street.jpg", DATA / "fruits.jpg"]
paths = [p for p in paths if p.exists()]
fig, axes = plt.subplots(len(paths), 2, figsize=(14, 5*len(paths)))
if len(paths) == 1: axes = axes[None, :]
for row, p in enumerate(paths):
    yb, yl, yc = yolo_predict(p, 0.25)
    rb, rl, rc = rtdetr_predict(p, 0.3)
    show_boxes_pil(p, yb, yl, yc, color="lime", ax=axes[row][0])
    axes[row][0].set_title(f"YOLO11n — {p.name}  ({len(yb)} boxes)")
    show_boxes_pil(p, rb, rl, rc, color="cyan", ax=axes[row][1])
    axes[row][1].set_title(f"RT-DETR — {p.name}  ({len(rb)} boxes)")
plt.tight_layout(); plt.show()

### Speed comparison


In [ ]:
# Speed comparison (single image, T4 GPU)
sample = DATA / "street.jpg"

# YOLO warmup + timing
for _ in range(3): yolo_predict(sample)
t0 = time.time(); N = 30
for _ in range(N): yolo_predict(sample)
yolo_ms = (time.time() - t0) / N * 1000

# RT-DETR warmup + timing
for _ in range(3): rtdetr_predict(sample)
t0 = time.time();
for _ in range(N): rtdetr_predict(sample)
rt_ms = (time.time() - t0) / N * 1000

print(f"YOLO11n   : {yolo_ms:5.1f} ms / image  ({1000/yolo_ms:.0f} fps)")
print(f"RT-DETR v2 r18: {rt_ms:5.1f} ms / image  ({1000/rt_ms:.0f} fps)")

## 5. Part D — VLM detection with Gemma 4

The third paradigm: let a language model **write the boxes**. **Gemma 4** (Google DeepMind, 2025) is a general open multimodal model (image-text-to-text). Ask it to detect objects and it returns clean **JSON** — coordinates normalised to **0–1000** — which we parse with `json.loads`. No special tokens, no custom decoder.

> ⚠️ Use an **instruction-tuned `-it`** checkpoint (e.g. `google/gemma-4-E2B-it`). A *base* model (no `-it`) has no chat template and won't follow the prompt. The model is **gated** — request access on HF and log in below. First download is a few GB (cached for the session); we use the smallest **E2B-it** so it fits a T4.
>
> 💡 *Historical note:* the older **PaliGemma** pioneered grounded VLM output using special `<loc0000>`…`<loc1023>` tokens; Gemma 4's JSON is the friendlier successor. We run **only Gemma** here — loading two VLMs at once would exhaust the T4's memory.


In [ ]:
# Gemma 4 is gated — log in once with an HF token (uncomment), or run `huggingface-cli login` in a terminal.
# from huggingface_hub import login
# login(token=os.environ.get("HF_TOKEN", ""))

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

# Gemma 4 (Google DeepMind, 2025) — open multimodal (image-text-to-text). Gated: request access on HF.
# IMPORTANT: use an INSTRUCTION-TUNED checkpoint (suffix '-it'). Base models (e.g. google/gemma-4-E2B,
# no '-it') have NO chat template and do NOT follow prompts -> they can't do prompted detection.
# Instruction-tuned ids (smallest first; first download is several GB but cached per session):
#   google/gemma-4-E2B-it  ·  google/gemma-4-E4B-it     (small — run on a Colab T4)
#   google/gemma-4-26B-A4B-it  ·  google/gemma-4-31B-it (large — need a 24-80 GB GPU)
GEMMA_ID = "google/gemma-4-E2B-it"          # smallest / fastest download; bump to -E4B-it for quality
gproc  = AutoProcessor.from_pretrained(GEMMA_ID)
assert getattr(gproc, "chat_template", None), (
    f"{GEMMA_ID} has no chat template — pick an instruction-tuned '-it' checkpoint, not a base model.")
# Load directly onto ONE device with a CONCRETE device_map. Do NOT use device_map="auto" (it can
# offload parts to the 'meta' device -> .generate() crashes with "meta tensors") and do NOT load then
# .to(device) (leaves lazily-initialised vision buffers on 'meta'). A concrete map streams the weights
# straight to the GPU (no CPU-RAM doubling, which E2B needs on a 16 GB T4). fp16 keeps it T4-friendly.
_dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
_devmap = "cuda" if torch.cuda.is_available() else "cpu"
gmodel = AutoModelForMultimodalLM.from_pretrained(GEMMA_ID, dtype=_dtype, device_map=_devmap).eval()
print("Loaded", GEMMA_ID, "on", _devmap)

**Parsing the JSON.** Gemma returns a list like `[{"box_2d": [ymin, xmin, ymax, xmax], "label": "apple"}, …]` with coords in **0–1000**. We grab the JSON list (ignoring any prose) and convert to pixel boxes.

In [ ]:
import re, json   # re/json also imported here so the Gemma cells run standalone

def parse_gemma_json(text, img_size):
    """Parse Gemma's JSON detections → [(label, [x1,y1,x2,y2])] in pixel coords."""
    W, H = img_size
    m = re.search(r"\[.*\]", text, re.DOTALL)          # grab the JSON list, ignore prose / ``` fences
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    out = []
    for d in data:
        if "box_2d" not in d:
            continue
        ymin, xmin, ymax, xmax = d["box_2d"]            # Gemma 4 order: y_min, x_min, y_max, x_max in 0-1000
        box = [xmin/1000*W, ymin/1000*H, xmax/1000*W, ymax/1000*H]
        out.append((d.get("label", "?"), box))
    return out

In [ ]:
def gemma_detect(image_path, classes="apple, banana, orange, fruit", max_new_tokens=512, image_tokens=560):
    img = Image.open(image_path).convert("RGB")
    prompt = (f"Detect these objects: {classes}. "
              'Return ONLY a JSON list; each item is '
              '{"box_2d": [ymin, xmin, ymax, xmax], "label": <name>} '
              "with coordinates normalised to 0-1000.")
    # Gemma 4 wants the image BEFORE the text; disable thinking so the reply is clean JSON.
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": prompt}]}]
    # image_tokens = visual token budget (70 / 140 / 280 / 560 / 1120). Higher -> more detail for
    # small objects/OCR but slower; lower -> faster. 560 is a good middle ground for detection.
    inputs = gproc.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", enable_thinking=False,
        processor_kwargs={"max_soft_tokens": image_tokens}).to(device)
    in_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = gmodel.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = gproc.decode(out[0][in_len:], skip_special_tokens=True)
    return text, parse_gemma_json(text, img.size)

# Pick a test image (defined here too so this cell runs standalone after a restart)
target_img = DATA / "fruits.jpg"   # fruit scene → matches the fruit classes we ask Gemma for

raw, dets = gemma_detect(target_img)
print("Raw Gemma 4 output:\n", raw)
print(f"\nParsed {len(dets)} detections.")

In [ ]:
# Visualise the Gemma JSON detections
if dets:
    boxes = [b for _, b in dets]; labels = [l for l, _ in dets]
    ax = show_boxes_pil(target_img, boxes, labels, color="#9333EA")
    ax.set_title("Gemma 4 — JSON-based open-vocabulary detection")
    plt.show()
else:
    print("No JSON parsed. Print `raw` above to see what the model returned.")

### Open-vocabulary / referring expressions

The real super-power of a VLM: you don't need a fixed class list — just *describe* what you want, even with attributes or spatial relations. The model never saw "the leftmost fruit" as a label class; it pieces the meaning together from pretraining.

In [ ]:
raw, dets = gemma_detect(target_img, classes="the leftmost fruit")
print("Raw:", raw[:200])
if dets:
    boxes = [b for _, b in dets]; labels = [l for l, _ in dets]
    ax = show_boxes_pil(target_img, boxes, labels, color="#0891B2")
    ax.set_title("Gemma 4 — open-vocabulary: 'the leftmost fruit'")
    plt.show()
else:
    print("No detection — try rephrasing the description.")

## 6. Detectors compared

| Aspect | YOLO11n | RT-DETR v2 (r18) | Gemma 4 (VLM) |
|--------|---------|------------------|----------------------------|
| Architecture | CNN, anchor-free, decoupled head | Transformer, set prediction | ViT + LM, autoregressive |
| Output | bbox + class + score | bbox + class + score | text: **JSON** (`box_2d` in 0–1000) |
| NMS | yes | **no** (1-to-1 matching) | n/a (generated one at a time) |
| Open vocabulary | no | no | **yes** |
| Speed (T4) | ~100 fps | ~30 fps | ~1 generation / sec |
| Fine-tuning | seconds / minutes | minutes | tricky (LoRA/PEFT needed) |
| Where it wins | real-time, on-device, fixed classes | best of both, no NMS, transformer flexibility | zero-shot, natural-language conditioning |

The from-scratch detector from **Part A** isn't in the table — it's a teaching tool, not a contender. But it's the conceptual ancestor of the YOLO column: same grid, anchors, regression, and NMS, just without the engineering.

> If you remember just one thing: **for the BAMBI project, start with YOLO11**. If you need *exotic* / unlabelled classes, *then* reach for a VLM.


## ✏️ Exercises

1. **Fine-tune YOLO on real data.** Replace the synthetic pen dataset with a small *real* set: take 20–30 photos of pens (or any object) on a desk, label them in [Label Studio](https://labelstud.io) or [Roboflow](https://roboflow.com), export YOLO format, and re-train. How does mAP — and the usable confidence threshold — compare to the synthetic baseline?

2. **Match the same image with YOLO and RT-DETR.** For three of the supplied images, compute IoU between each pair of YOLO ↔ RT-DETR detections. Which classes do the two models *both* detect? Which does one find but the other misses? Plot a confusion-style matrix.

3. **Make the VLM count — and probe its limits.** Prompt Gemma 4 to detect a countable class (e.g. apples) on 10 images and compare its count to YOLO's. Then try open-vocabulary queries ("the ripest banana", "the object top-left"). Where does it shine, and where does it miss — small objects, crowded scenes, malformed JSON?

4. **Extend the from-scratch detector (Part A).** Give each cell **two** anchors of different aspect ratio (e.g. tall vs. wide) and assign each GT to the anchor with the highest IoU. Does it help when fruits are pasted as non-square crops? Then add a tiny **class head** (apple / banana / …) and turn it into a multi-class detector.


In [ ]:
# Your code here